# Processed UMAPs — rasterized points, vector text

Plots the UMAP already stored in `inprogress_3.h5ad` (`obsm['X_umap']`), exported once per
colouring as its own SVG: `batch`, `leiden_vi`, `annot_5`, `annot_8_c2l`, `clinical_outcome`.

The scatter points are **rasterized** so each SVG stays light with ~95k cells, while axes, title,
tick labels and the legend remain editable vector/text objects in Illustrator or Inkscape.

The h5ad is opened strictly read-only and only `obs`/`obsm`/`uns` are loaded — `X` (8 GB, dense)
and `layers` are never touched.

In [ ]:
import os
# Project data root. Override with the ACC_DATA_ROOT environment variable.
ACC_DATA_ROOT = os.environ.get("ACC_DATA_ROOT", os.path.expanduser("~/acc_data"))


In [ ]:
from pathlib import Path

import anndata as ad
import h5py
import matplotlib as mpl
import matplotlib.pyplot as plt
import scanpy as sc

# 'none' keeps every label as a real <text> element in the SVG instead of outlined paths
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["pdf.fonttype"] = 42
sc.set_figure_params(vector_friendly=True, dpi_save=600, frameon=False)

H5 = Path(f"{ACC_DATA_ROOT}/outputs/h5ad_files/inprogress_3.h5ad")
OUT = Path(f"{ACC_DATA_ROOT}/outputs/images")
COLOR_KEYS = ["batch", "leiden_vi", "annot_5", "annot_8_c2l", "clinical_outcome", 'annot_6']

In [ ]:
# Slim read: obs + X_umap + the saved palettes only. 'r' mode never writes to the file.
try:
    read_elem = ad.io.read_elem
except AttributeError:
    from anndata.experimental import read_elem

palettes = {}
with h5py.File(H5, "r") as f:
    obs = read_elem(f["obs"])
    umap = read_elem(f["obsm/X_umap"])
    for key in COLOR_KEYS:
        src = f"uns/{key}_colors"
        if src not in f:
            # clinical_outcome has no saved palette; borrow the Good/Poor colours used by
            # 'outcome' so it matches pca_by_clinical_outcome.svg
            src = "uns/outcome_colors"
        if src in f:
            palettes[key] = list(read_elem(f[src]))

adata = ad.AnnData(obs=obs, obsm={"X_umap": umap})
for key, colors in palettes.items():
    adata.uns[f"{key}_colors"] = colors

print(f"{adata.n_obs:,} cells")
for key in COLOR_KEYS:
    print(f"  {key}: {adata.obs[key].nunique()} groups, palette loaded: {key in palettes}")

In [ ]:
def export_umap(key):
    """Plot the UMAP coloured by `key` and write it as an SVG with rasterized dots."""
    fig, ax = plt.subplots(figsize=(7, 6))

    sc.pl.umap(
        adata,
        color=key,
        legend_loc="right margin",
        size=6,
        frameon=False,
        ax=ax,
        show=False,
    )

    # Rasterize the point cloud only. Collections are the scatter dots; text, spines and the
    # legend are not collections, so they are left as vector objects.
    for coll in ax.collections:
        coll.set_rasterized(True)

    out = OUT / f"processed_umap_{key}.svg"
    # dpi controls the resolution of the rasterized dot layer inside the SVG
    fig.savefig(out, dpi=600, bbox_inches="tight")
    plt.close(fig)
    return out


written = [export_umap(key) for key in COLOR_KEYS]
for path in written:
    print(f"{path.name}  ({path.stat().st_size / 1e6:.2f} MB)")

In [ ]:
# Sanity check: in every SVG the dots must be one embedded raster image and the labels
# must still be text.
for path in written:
    svg = path.read_text()
    n_image = svg.count("<image")
    n_text = svg.count("<text")
    print(f"{path.name:40s} raster layers: {n_image}   editable <text>: {n_text}")
    assert n_image >= 1, f"{path.name}: points were not rasterized"
    assert n_text >= 1, f"{path.name}: labels were outlined instead of kept as text"

print("\nOK — all SVGs have rasterized points and editable text")